In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import glob
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on: {device}")

# Set this to the directory containing the project data.
BASE_DIR = "/content/drive/MyDrive/Babol_Segmentation"

PATCH_IMG_DIR = os.path.join(BASE_DIR, "patches", "images")
PATCH_MASK_DIR = os.path.join(BASE_DIR, "patches", "masks")
MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

BASELINE_MODEL_PATH = os.path.join(
    MODEL_DIR, "best_unet_model.pth"
)

FINE_TUNED_MODEL_PATH = os.path.join(
    MODEL_DIR, "fine_tuned_babol_unet_best.pth"
)

CLASS_NAMES = [
    "Background",
    "Plants",
    "Buildings",
    "Water",
    "Roads"
]

# RGB colors used to display predicted segmentation masks.
CLASS_COLORS = np.array([
    [0, 0, 0],        # Background
    [0, 128, 0],      # Plants
    [255, 0, 0],      # Buildings
    [0, 0, 255],      # Water
    [255, 255, 0]     # Roads
], dtype=np.uint8)

NUM_CLASSES = len(CLASS_NAMES)

print("Project directories configured.")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
def get_paired_paths(image_dir, mask_dir):
    image_paths = sorted(
        glob.glob(os.path.join(image_dir, "*.png"))
    )

    mask_paths = sorted(
        glob.glob(os.path.join(mask_dir, "*.png"))
    )

    mask_lookup = {
        os.path.basename(path): path
        for path in mask_paths
    }

    paired_images = []
    paired_masks = []

    for image_path in image_paths:
        filename = os.path.basename(image_path)

        if filename in mask_lookup:
            paired_images.append(image_path)
            paired_masks.append(mask_lookup[filename])

    return paired_images, paired_masks


paired_images, paired_masks = get_paired_paths(
    PATCH_IMG_DIR,
    PATCH_MASK_DIR
)

print(f"Available paired patches: {len(paired_images)}")

if len(paired_images) == 0:
    raise RuntimeError(
        "No matching image-mask pairs were found. "
        "Check the configured dataset directories."
    )

NUM_SAMPLES = min(500, len(paired_images))

random.seed(SEED)

selected_indices = random.sample(
    range(len(paired_images)),
    NUM_SAMPLES
)

sampled_images = [
    paired_images[index]
    for index in selected_indices
]

sampled_masks = [
    paired_masks[index]
    for index in selected_indices
]

print(f"Selected {len(sampled_images)} paired patches.")

In [ ]:
class BabolPatchDataset(Dataset):

    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):

        image_path = self.image_paths[index]
        mask_path = self.mask_paths[index]

        image = cv2.imread(image_path)

        if image is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            mask_path,
            cv2.IMREAD_GRAYSCALE
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {mask_path}"
            )

        image = image.astype(np.float32) / 255.0

        image_tensor = torch.tensor(
            image,
            dtype=torch.float32
        ).permute(2, 0, 1)

        mask_tensor = torch.tensor(
            mask,
            dtype=torch.long
        )

        return image_tensor, mask_tensor


dataset = BabolPatchDataset(
    sampled_images,
    sampled_masks
)

train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"Dataset size: {len(dataset)}")
print(f"Training batches: {len(train_loader)}")

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=5):
        super().__init__()

        # Encoder
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.conv1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        # Bottleneck
        b = self.bottleneck(p4)

        # Decoder with skip connections
        u4 = self.up4(b)
        c4 = self.conv4(
            torch.cat([d4, u4], dim=1)
        )

        u3 = self.up3(c4)
        c3 = self.conv3(
            torch.cat([d3, u3], dim=1)
        )

        u2 = self.up2(c3)
        c2 = self.conv2(
            torch.cat([d2, u2], dim=1)
        )

        u1 = self.up1(c2)
        c1 = self.conv1(
            torch.cat([d1, u1], dim=1)
        )

        return self.final_conv(c1)


model = UNet(
    in_channels=3,
    out_channels=NUM_CLASSES
).to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

NUM_EPOCHS = 30

history_loss = []
best_loss = float("inf")

print(f"Starting baseline training on {device}.")

for epoch in range(1, NUM_EPOCHS + 1):

    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{NUM_EPOCHS}"
    )

    for images, masks in progress_bar:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            masks
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        progress_bar.set_postfix({
            "Batch Loss": f"{loss.item():.4f}"
        })

    epoch_loss = running_loss / len(dataset)
    history_loss.append(epoch_loss)

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            model.state_dict(),
            BASELINE_MODEL_PATH
        )

        save_message = "Best weights saved."

    else:
        save_message = ""

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] "
        f"Loss: {epoch_loss:.4f} {save_message}"
    )

print(f"Training completed. Best loss: {best_loss:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    history_loss,
    marker="o"
)

plt.title("U-Net Baseline Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")

plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
model = UNet(
    in_channels=3,
    out_channels=NUM_CLASSES
).to(device)

if not os.path.exists(BASELINE_MODEL_PATH):
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {BASELINE_MODEL_PATH}"
    )

model.load_state_dict(
    torch.load(
        BASELINE_MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Baseline U-Net checkpoint loaded.")

In [ ]:
def enhance_clahe_sharpen(img_bgr):
    """
    Apply CLAHE to improve local contrast and unsharp masking
    to emphasize spatial details.
    """

    lab = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2LAB
    )

    lightness, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )

    enhanced_lightness = clahe.apply(lightness)

    enhanced_lab = cv2.merge((
        enhanced_lightness,
        a,
        b
    ))

    enhanced_bgr = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )

    gaussian = cv2.GaussianBlur(
        enhanced_bgr,
        (0, 0),
        sigmaX=2.0
    )

    sharpened = cv2.addWeighted(
        enhanced_bgr,
        1.5,
        gaussian,
        -0.5,
        0
    )

    return sharpened


def bicubic_upsample(img_bgr, scale_factor=2.0):
    """
    Upsample an image using PyTorch bicubic interpolation.
    """

    img_rgb = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2RGB
    )

    tensor = torch.from_numpy(
        img_rgb
    ).permute(2, 0, 1).unsqueeze(0).float()

    upsampled = F.interpolate(
        tensor,
        scale_factor=scale_factor,
        mode="bicubic",
        align_corners=False
    )

    output_rgb = (
        upsampled.squeeze(0)
        .permute(1, 2, 0)
        .clamp(0, 255)
        .byte()
        .cpu()
        .numpy()
    )

    return cv2.cvtColor(
        output_rgb,
        cv2.COLOR_RGB2BGR
    )

In [ ]:
from skimage.exposure import match_histograms

YEARS = ["2006", "2012", "2019", "2025"]

def load_babol_year_images(base_dir, years):
    images = {}

    for year in years:

        image_path = os.path.join(
            base_dir,
            f"cropped{year}.png"
        )

        if not os.path.exists(image_path):
            print(f"Image not found for {year}: {image_path}")
            continue

        image_bgr = cv2.imread(image_path)

        if image_bgr is None:
            print(f"Could not read image for {year}.")
            continue

        images[year] = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB
        )

    return images


year_images = load_babol_year_images(
    BASE_DIR,
    YEARS
)

if "2025" not in year_images:
    raise FileNotFoundError(
        "The 2025 reference image is required for calibration."
    )

reference_image = year_images["2025"]

calibrated_images = {}

for year, image in year_images.items():

    if year == "2025":
        calibrated_images[year] = reference_image.copy()

    else:
        calibrated_images[year] = match_histograms(
            image,
            reference_image,
            channel_axis=-1
        ).astype(np.uint8)

print("Histogram calibration completed.")

In [ ]:
available_years = list(calibrated_images.keys())

fig, axes = plt.subplots(
    2,
    len(available_years),
    figsize=(4 * len(available_years), 8)
)

for index, year in enumerate(available_years):

    axes[0, index].imshow(year_images[year])
    axes[0, index].set_title(f"{year} Original")
    axes[0, index].axis("off")

    axes[1, index].imshow(calibrated_images[year])
    axes[1, index].set_title(f"{year} Calibrated")
    axes[1, index].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
PATCH_SIZE = 256

ENHANCED_PATCH_DIR = os.path.join(
    BASE_DIR,
    "babol_patches_enhanced",
    "images"
)

os.makedirs(
    ENHANCED_PATCH_DIR,
    exist_ok=True
)

enhanced_images = {}

for year, image_rgb in calibrated_images.items():

    image_bgr = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2BGR
    )

    upsampled_bgr = bicubic_upsample(
        image_bgr,
        scale_factor=2.0
    )

    enhanced_bgr = enhance_clahe_sharpen(
        upsampled_bgr
    )

    enhanced_rgb = cv2.cvtColor(
        enhanced_bgr,
        cv2.COLOR_BGR2RGB
    )

    enhanced_images[year] = enhanced_rgb

    year_output_dir = os.path.join(
        ENHANCED_PATCH_DIR,
        year
    )

    os.makedirs(
        year_output_dir,
        exist_ok=True
    )

    height, width, _ = enhanced_rgb.shape

    patch_count = 0

    for y in range(
        0,
        height - PATCH_SIZE + 1,
        PATCH_SIZE
    ):

        for x in range(
            0,
            width - PATCH_SIZE + 1,
            PATCH_SIZE
        ):

            patch = enhanced_rgb[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            patch_path = os.path.join(
                year_output_dir,
                f"patch_{patch_count:04d}.png"
            )

            cv2.imwrite(
                patch_path,
                cv2.cvtColor(
                    patch,
                    cv2.COLOR_RGB2BGR
                )
            )

            patch_count += 1

    print(
        f"{year}: generated {patch_count} patches."
    )

In [ ]:
class CombinedFocalDiceLoss(nn.Module):

    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()

        self.gamma = gamma

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=alpha,
            reduction="none"
        )

    def forward(self, inputs, targets):

        # Focal loss
        log_pt = -self.cross_entropy(
            inputs,
            targets
        )

        pt = torch.exp(log_pt)

        focal_loss = (
            -((1 - pt) ** self.gamma) * log_pt
        ).mean()

        # Multiclass Dice loss
        num_classes = inputs.shape[1]

        probabilities = F.softmax(
            inputs,
            dim=1
        )

        targets_onehot = F.one_hot(
            targets,
            num_classes=num_classes
        ).permute(0, 3, 1, 2).float()

        intersection = torch.sum(
            probabilities * targets_onehot,
            dim=(0, 2, 3)
        )

        cardinality = torch.sum(
            probabilities + targets_onehot,
            dim=(0, 2, 3)
        )

        dice_loss = 1.0 - torch.mean(
            (2.0 * intersection + 1e-6) /
            (cardinality + 1e-6)
        )

        return focal_loss + dice_loss

In [ ]:
class FineTuneDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_paths = sorted(
            glob.glob(
                os.path.join(image_dir, "*.*")
            )
        )

        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):

        image_path = self.image_paths[index]
        filename = os.path.basename(image_path)

        mask_path = os.path.join(
            self.mask_dir,
            filename
        )

        if not os.path.exists(mask_path):

            base_name = os.path.splitext(filename)[0]

            mask_path = os.path.join(
                self.mask_dir,
                f"{base_name}.png"
            )

        image = cv2.imread(image_path)

        if image is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            mask_path,
            cv2.IMREAD_GRAYSCALE
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {mask_path}"
            )

        image = image.astype(np.float32) / 255.0

        image_tensor = torch.tensor(
            image,
            dtype=torch.float32
        ).permute(2, 0, 1)

        mask_tensor = torch.tensor(
            mask,
            dtype=torch.long
        )

        return image_tensor, mask_tensor

In [ ]:
PATCH_IMG_DIR = os.path.join(
    BASE_DIR,
    "patches",
    "images"
)

PATCH_MASK_DIR = os.path.join(
    BASE_DIR,
    "patches",
    "masks"
)

fine_tune_dataset = FineTuneDataset(
    PATCH_IMG_DIR,
    PATCH_MASK_DIR
)

fine_tune_loader = DataLoader(
    fine_tune_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

model = UNet(
    in_channels=3,
    out_channels=NUM_CLASSES
).to(device)

if not os.path.exists(BASELINE_MODEL_PATH):
    raise FileNotFoundError(
        "The baseline model must be trained before fine-tuning."
    )

model.load_state_dict(
    torch.load(
        BASELINE_MODEL_PATH,
        map_location=device
    )
)

class_weights = torch.tensor(
    [0.5, 1.0, 3.5, 1.5, 2.0],
    dtype=torch.float32
).to(device)

criterion = CombinedFocalDiceLoss(
    gamma=2.0,
    alpha=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

EPOCHS = 15
best_loss = float("inf")

fine_tuning_history = []

print("Starting U-Net fine-tuning with Focal + Dice loss.")

for epoch in range(1, EPOCHS + 1):

    model.train()
    running_loss = 0.0

    for images, masks in tqdm(
        fine_tune_loader,
        desc=f"Fine-tuning {epoch}/{EPOCHS}"
    ):

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            masks
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(fine_tune_loader)

    fine_tuning_history.append(epoch_loss)

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            model.state_dict(),
            FINE_TUNED_MODEL_PATH
        )

        print("Best fine-tuned checkpoint saved.")

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"Loss: {epoch_loss:.4f}"
    )

print(
    f"Fine-tuning completed. "
    f"Lowest training loss: {best_loss:.4f}"
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, EPOCHS + 1),
    fine_tuning_history,
    marker="o"
)

plt.title("U-Net Fine-Tuning: Focal + Dice Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
model = UNet(
    in_channels=3,
    out_channels=NUM_CLASSES
).to(device)

if not os.path.exists(FINE_TUNED_MODEL_PATH):
    raise FileNotFoundError(
        f"Fine-tuned model not found: {FINE_TUNED_MODEL_PATH}"
    )

model.load_state_dict(
    torch.load(
        FINE_TUNED_MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Fine-tuned U-Net loaded successfully.")

In [ ]:
def compute_iou_dice(predictions, targets, num_classes=5):

    ious = []
    dices = []

    for class_id in range(num_classes):

        pred_class = predictions == class_id
        target_class = targets == class_id

        intersection = np.logical_and(
            pred_class,
            target_class
        ).sum()

        union = np.logical_or(
            pred_class,
            target_class
        ).sum()

        iou = (
            (intersection + 1e-6) /
            (union + 1e-6)
        )

        dice = (
            (2.0 * intersection + 1e-6) /
            (
                pred_class.sum() +
                target_class.sum() +
                1e-6
            )
        )

        ious.append(iou * 100)
        dices.append(dice * 100)

    return ious, dices


model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for images, masks in fine_tune_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        ).cpu().numpy()

        all_predictions.append(predictions)
        all_targets.append(masks.numpy())


all_predictions = np.concatenate(
    all_predictions,
    axis=0
)

all_targets = np.concatenate(
    all_targets,
    axis=0
)

ious, dices = compute_iou_dice(
    all_predictions,
    all_targets,
    num_classes=NUM_CLASSES
)

mean_iou = np.mean(ious)

metrics_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "IoU (%)": np.round(ious, 2),
    "Dice (%)": np.round(dices, 2)
})

print(metrics_df.to_string(index=False))
print(f"\nMean IoU: {mean_iou:.2f}%")

In [ ]:
def predict_image(image_rgb, model, device):

    image = image_rgb.astype(np.float32) / 255.0

    image_tensor = torch.tensor(
        image,
        dtype=torch.float32
    ).permute(2, 0, 1).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image_tensor)

        prediction = torch.argmax(
            outputs,
            dim=1
        ).squeeze(0).cpu().numpy()

    return prediction


def colorize_mask(prediction):

    return CLASS_COLORS[prediction]


def calculate_class_percentages(prediction):

    total_pixels = prediction.size

    return {
        CLASS_NAMES[class_id]: round(
            (
                np.sum(prediction == class_id) /
                total_pixels
            ) * 100,
            2
        )
        for class_id in range(NUM_CLASSES)
    }

In [ ]:
multi_temporal_results = {}
multi_temporal_predictions = {}

for year in YEARS:

    if year not in calibrated_images:
        continue

    image_rgb = calibrated_images[year]

    prediction = predict_image(
        image_rgb,
        model,
        device
    )

    percentages = calculate_class_percentages(
        prediction
    )

    multi_temporal_results[year] = percentages
    multi_temporal_predictions[year] = prediction

    color_mask = colorize_mask(prediction)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5)
    )

    axes[0].imshow(image_rgb)
    axes[0].set_title(f"Babol {year} Original")
    axes[0].axis("off")

    axes[1].imshow(color_mask)
    axes[1].set_title(f"Babol {year} U-Net Prediction")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


distribution_df = pd.DataFrame(
    multi_temporal_results
)

print("Babol Land-Cover Distribution (%)")
print(distribution_df.to_string())

In [ ]:
distribution_data = {
    "Year": [2006, 2012, 2019, 2025],
    "Background": [35.33, 36.55, 33.90, 40.14],
    "Plants": [28.43, 30.74, 30.28, 34.91],
    "Buildings": [17.18, 11.49, 15.34, 8.98],
    "Water": [0.23, 0.79, 0.21, 0.46],
    "Roads": [18.82, 20.44, 20.27, 15.51]
}

distribution_df = pd.DataFrame(
    distribution_data
)

plt.figure(figsize=(10, 6))

plt.plot(
    distribution_df["Year"],
    distribution_df["Buildings"],
    marker="o",
    linewidth=2.5,
    label="Buildings"
)

plt.plot(
    distribution_df["Year"],
    distribution_df["Plants"],
    marker="s",
    linewidth=2.5,
    label="Plants"
)

plt.plot(
    distribution_df["Year"],
    distribution_df["Roads"],
    marker="^",
    linewidth=2.5,
    label="Roads"
)

plt.plot(
    distribution_df["Year"],
    distribution_df["Water"],
    marker="d",
    linewidth=2.5,
    label="Water"
)

plt.title("Babol Land-Cover Dynamics (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Coverage Area (%)")

plt.xticks([2006, 2012, 2019, 2025])

plt.grid(
    True,
    linestyle="--",
    alpha=0.6
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
REGION_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "4years"
)

CLASS_COLORS = np.array([
    [0, 0, 0],
    [34, 139, 34],
    [255, 0, 0],
    [0, 0, 255],
    [255, 255, 0]
], dtype=np.uint8)


all_files = sorted(
    glob.glob(
        os.path.join(REGION_IMAGE_DIR, "*.*")
    )
)

regions = {}

for image_path in all_files:

    filename = os.path.basename(image_path)
    parts = filename.split(".")

    if len(parts) < 3:
        continue

    region_id = parts[0]
    year = parts[1]

    regions.setdefault(
        region_id,
        []
    ).append((year, image_path))


for region_id, year_files in regions.items():

    year_files = sorted(
        year_files,
        key=lambda item: item[0]
    )

    reference_bgr = cv2.imread(
        year_files[-1][1]
    )

    reference_rgb = cv2.cvtColor(
        reference_bgr,
        cv2.COLOR_BGR2RGB
    )

    fig, axes = plt.subplots(
        2,
        len(year_files),
        figsize=(4 * len(year_files), 8)
    )

    if len(year_files) == 1:
        axes = np.expand_dims(axes, axis=1)

    region_statistics = {}

    for index, (year, image_path) in enumerate(year_files):

        image_bgr = cv2.imread(image_path)

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB
        )

        if year != year_files[-1][0]:

            calibrated = match_histograms(
                image_rgb,
                reference_rgb,
                channel_axis=-1
            ).astype(np.uint8)

        else:
            calibrated = reference_rgb

        height, width, _ = calibrated.shape

        target_height = (height // 32) * 32
        target_width = (width // 32) * 32

        resized = cv2.resize(
            calibrated,
            (target_width, target_height)
        )

        prediction = predict_image(
            resized,
            model,
            device
        )

        region_statistics[year] = (
            calculate_class_percentages(prediction)
        )

        axes[0, index].imshow(resized)
        axes[0, index].set_title(
            f"Region {region_id} - {year}"
        )
        axes[0, index].axis("off")

        axes[1, index].imshow(
            colorize_mask(prediction)
        )
        axes[1, index].set_title(
            f"Segmentation - {year}"
        )
        axes[1, index].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Region {region_id} land-cover distribution:")
    print(
        pd.DataFrame(region_statistics).to_string()
    )

In [ ]:
FULL_CITY_INPUT_DIR = os.path.join(
    BASE_DIR,
    "full_city_raw"
)

FULL_CITY_OUTPUT_DIR = os.path.join(
    RESULTS_DIR,
    "full_city_segmented"
)

os.makedirs(
    FULL_CITY_OUTPUT_DIR,
    exist_ok=True
)

PATCH_SIZE = 256


def segment_full_image(
    image_path,
    model,
    patch_size=256
):

    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        raise RuntimeError(
            f"Could not read image: {image_path}"
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    height, width, _ = image_rgb.shape

    pad_height = (
        patch_size - height % patch_size
    ) % patch_size

    pad_width = (
        patch_size - width % patch_size
    ) % patch_size

    padded_image = np.pad(
        image_rgb,
        (
            (0, pad_height),
            (0, pad_width),
            (0, 0)
        ),
        mode="constant"
    )

    padded_height, padded_width, _ = (
        padded_image.shape
    )

    full_mask = np.zeros(
        (padded_height, padded_width),
        dtype=np.uint8
    )

    model.eval()

    with torch.no_grad():

        for y in range(
            0,
            padded_height,
            patch_size
        ):

            for x in range(
                0,
                padded_width,
                patch_size
            ):

                patch = padded_image[
                    y:y + patch_size,
                    x:x + patch_size
                ]

                tensor = torch.tensor(
                    patch,
                    dtype=torch.float32
                ).permute(2, 0, 1).unsqueeze(0)

                tensor = tensor.to(device) / 255.0

                output = model(tensor)

                prediction = torch.argmax(
                    output,
                    dim=1
                ).squeeze(0).cpu().numpy()

                full_mask[
                    y:y + patch_size,
                    x:x + patch_size
                ] = prediction

    # Remove padding to restore the original image dimensions.
    full_mask = full_mask[:height, :width]

    color_mask = colorize_mask(full_mask)

    return full_mask, color_mask

In [ ]:
city_image_paths = []

for extension in ["*.png", "*.jpg", "*.jpeg"]:
    city_image_paths.extend(
        glob.glob(
            os.path.join(
                FULL_CITY_INPUT_DIR,
                extension
            )
        )
    )

city_image_paths = sorted(city_image_paths)

print(
    f"Found {len(city_image_paths)} full-city images."
)

for image_path in city_image_paths:

    filename = os.path.basename(image_path)

    print(f"Processing: {filename}")

    prediction, color_mask = segment_full_image(
        image_path,
        model,
        patch_size=PATCH_SIZE
    )

    output_path = os.path.join(
        FULL_CITY_OUTPUT_DIR,
        f"segmented_{filename}"
    )

    cv2.imwrite(
        output_path,
        cv2.cvtColor(
            color_mask,
            cv2.COLOR_RGB2BGR
        )
    )

    print(f"Saved: {output_path}")

print("Full-city segmentation completed.")

In [ ]:
segmented_files = []

for extension in ["*.png", "*.jpg", "*.jpeg"]:
    segmented_files.extend(
        glob.glob(
            os.path.join(
                FULL_CITY_OUTPUT_DIR,
                extension
            )
        )
    )

city_statistics = {}

for file_path in sorted(segmented_files):

    image_bgr = cv2.imread(file_path)

    if image_bgr is None:
        continue

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    total_pixels = image_rgb.shape[0] * image_rgb.shape[1]

    statistics = {}

    for class_id, class_name in enumerate(CLASS_NAMES):

        color = CLASS_COLORS[class_id]

        matching_pixels = np.all(
            image_rgb == color,
            axis=-1
        )

        percentage = (
            np.sum(matching_pixels) /
            total_pixels
        ) * 100

        statistics[class_name] = round(
            percentage,
            2
        )

    filename = os.path.basename(file_path)

    city_statistics[filename] = statistics


city_statistics_df = pd.DataFrame(
    city_statistics
)

print("Full-city land-cover comparison (%)")
print(city_statistics_df.to_string())